# E1c — where does the gaze accuracy actually come from?

The live prototype fails on an unseen webcam: the model's predicted x has correlation
**-0.007** with the dot the person is looking at, and a refit on that person's own frames
recovers accuracy **only from the head-pose features** (head-only 0.54, eye-only 0.22,
chance 0.33).

That raises a question about E1b's headline number. If the 0.84 three-zone accuracy on EOTT
came mostly from head pose rather than eye movement, then the accuracy does not belong to
*gaze* at all — and it would not transfer to a person who cannot move their head, which is the
population this system is for.

This notebook answers it by ablation on the same features E1b used: grouped cross-validation
by participant, three feature sets. **This is a falsification test of our own result.** Run it
before writing anything about E1b.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, glob
FEAT = '/content/drive/MyDrive/gazespeakzero/E1b/features'
print('exists:', os.path.isdir(FEAT), '| files:', len(glob.glob(f'{FEAT}/P_*_frames.csv')))

In [ ]:
!pip -q install scikit-learn pandas
import pandas as pd, numpy as np, glob
F = pd.concat([pd.read_csv(f) for f in sorted(glob.glob(f'{FEAT}/P_*_frames.csv'))], ignore_index=True)
if 'is_laptop' not in F.columns:
    F['is_laptop'] = (F['setup'].astype(str).str.lower() == 'laptop').astype(int) if 'setup' in F.columns else 1
m = (F.detected == 1) & F.tobii_x.notna()
F = F.loc[m].reset_index(drop=True)
print(F.shape, '| participants:', F.participant.nunique())

## The three feature sets

- **all** — exactly what E1b and the exported model use (39 features).
- **eye** — eye landmarks and the eye blendshapes only. This is what a person who cannot move
  their head can produce.
- **head** — head rotation, translation and nose position only. **No eye information at all.**

If `head` alone comes close to `all`, the published number is a head-pose number.

In [ ]:
EYE = ['reye_h','reye_v','reye_vlid','reye_open','leye_h','leye_v','leye_vlid','leye_open']
BS  = ['bs_eyeLookInLeft','bs_eyeLookInRight','bs_eyeLookOutLeft','bs_eyeLookOutRight',
       'bs_eyeLookUpLeft','bs_eyeLookUpRight','bs_eyeLookDownLeft','bs_eyeLookDownRight',
       'bs_eyeBlinkLeft','bs_eyeBlinkRight','bs_eyeSquintLeft','bs_eyeSquintRight']
HEAD = ['R00','R01','R02','R10','R11','R12','R20','R21','R22','tx','ty','tz',
        'yaw','pitch','roll','nose_x','nose_y','interocular_norm']
SETS = {'all': EYE+BS+HEAD+['is_laptop'], 'eye': EYE+BS+['is_laptop'], 'head': HEAD+['is_laptop']}
for k, v in SETS.items():
    assert all(c in F.columns for c in v), (k, [c for c in v if c not in F.columns])
print({k: len(v) for k, v in SETS.items()})

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GroupKFold

def zone(v, z=3):
    return np.clip((np.clip(v, 0, 1) * z).astype(int), 0, z - 1)

groups = F.participant.values
y = F.tobii_x.values
res, per_p = {}, {}
gkf = GroupKFold(n_splits=5)
for name, cols in SETS.items():
    X = F[cols].values
    pred = np.full(len(F), np.nan)
    for tr, te in gkf.split(X, y, groups):
        r = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.06,
                                          early_stopping=False, random_state=0).fit(X[tr], y[tr])
        pred[te] = r.predict(X[te])
    ok = zone(pred) == zone(y)
    res[name] = float(ok.mean())
    per_p[name] = pd.Series(ok).groupby(groups).mean()
    print(f'{name:>5}: 1x3 zone accuracy {res[name]:.3f}   (held-out participants)')

## Does accuracy track iris resolution?

The live prototype failed completely at 640x480 (eye 33 px wide, iris signal-to-noise 0.5,
correlation with the target -0.007) and worked at 1280x720 (eye 55 px, signal-to-noise 7-8,
correlation +0.88). If that is a general law rather than one laptop's quirk, it should show up
inside EOTT: participants whose eyes occupy more pixels should be predicted better.

If it holds, capture resolution is a reporting requirement for this literature, and a webcam
gaze number without it is not reproducible.

In [ ]:
ew = [c for c in ('reye_width_px', 'leye_width_px') if c in F.columns]
if not ew:
    print('no eye-width columns in these features - skip this cell')
else:
    F['eye_px'] = F[ew].mean(axis=1)
    pp = pd.DataFrame({'eye_px': F.groupby('participant').eye_px.mean(),
                       'accuracy': per_p['all'],
                       'eye_accuracy': per_p['eye'],
                       'head_accuracy': per_p['head']}).dropna()
    print(pp.round(3).sort_values('eye_px').to_string())
    print()
    for col in ('accuracy', 'eye_accuracy', 'head_accuracy'):
        r = pp.eye_px.corr(pp[col])
        print('correlation(eye width in px, %s) over participants = %+.3f' % (col, r))
    print()
    q = pd.qcut(pp.eye_px, 3, labels=['smallest third', 'middle third', 'largest third'])
    print(pp.groupby(q, observed=False).mean().round(3).to_string())
    pp.to_csv('/content/drive/MyDrive/gazespeakzero/E1b/resolution_vs_accuracy.csv')
    print()
    print('saved resolution_vs_accuracy.csv - send me this one too')

## Read the result

- `head` near `all` → **E1b's number is head pose, not gaze.** The paper must say so, and the
  system's claim for people who cannot move their head is not supported. This becomes the
  paper's central finding rather than a limitation.
- `eye` near `all`, `head` near chance → the features are fine and the live failure is a
  camera/domain problem, which the per-session refit should then fix using eye features.
- both well below `all` → the two are needed jointly; report that.

In [ ]:
import numpy as np
summary = pd.DataFrame({'feature_set': list(res), 'zone_accuracy_1x3': [round(v,4) for v in res.values()],
                        'gap_to_all': [round(res[k]-res['all'],4) for k in res]})
print(summary.to_string(index=False))
print('\nchance = 0.333 | E1b reported 0.867 per frame, 0.838 at a 0.5 s dwell')
pp = pd.DataFrame(per_p); pp.index.name = 'participant'
print('\nper-participant (first 10):'); print(pp.head(10).round(3).to_string())
summary.to_csv('/content/drive/MyDrive/gazespeakzero/E1b/ablation_summary.csv', index=False)
pp.to_csv('/content/drive/MyDrive/gazespeakzero/E1b/ablation_per_participant.csv')
print('\nsaved to the E1b folder on Drive - send me both CSVs')